In [ ]:
import os
import json
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import ast
import numpy as np

In [ ]:
def load_all_results(base_dir: str) -> pd.DataFrame:
    """
    从指定基础目录下的所有实验子目录中加载 result.json 文件，
    并将它们合并成一个 pandas DataFrame。
    """
    all_results_data = []

    if not os.path.isdir(base_dir):
        print(f"错误: 基础目录 '{base_dir}' 不存在。")
        return pd.DataFrame()

    # 遍历基础目录中的所有条目
    for exp_folder_name in os.listdir(base_dir):
        exp_path = os.path.join(base_dir, exp_folder_name)

        # 确保它是一个目录
        if os.path.isdir(exp_path):
            result_file_path = os.path.join(exp_path, "result.json")

            # 检查 result.json 是否存在
            if os.path.exists(result_file_path):
                try:
                    with open(result_file_path, "r", encoding="utf-8") as f:
                        result_data = json.load(f)
                        # 添加实验文件夹名称作为一列，方便识别
                        result_data["experiment"] = exp_folder_name
                        all_results_data.append(result_data)
                except json.JSONDecodeError:
                    print(f"警告: 无法解析文件 {result_file_path} 中的 JSON。")
                except Exception as e:
                    print(f"警告: 读取文件 {result_file_path} 时出错: {e}")

    if not all_results_data:
        print("未找到或未能处理任何 'result.json' 文件。")
        return pd.DataFrame()

    # 将字典列表转换为 DataFrame
    results_df = pd.DataFrame(all_results_data)
    return results_df


In [ ]:
def cal_area_under_curve(df: pd.DataFrame, start:int, end:int) -> pd.DataFrame:
    """
    计算指定指标的曲线下面积(AUC)，并减去低于区间最小值的面积。
    同时更新 pass_k 与 pass_mean 为 end 步对应的准确率。
    
    参数:
        df: 包含实验结果的 DataFrame
        start: 起始 turn 数
        end: 结束 turn 数
    
    返回:
        添加了 AUC 与末端准确率列的 DataFrame
    """
    df = df.copy()
    
    passk_auc = []
    pass1_auc = []
    passk_end_values = []
    pass1_end_values = []
    
    for idx, row in df.iterrows():
        passk_dict = row['passk_at_T']
        pass1_dict = row['pass1_at_T']
        
        def extract_series(data_dict):
            values = []
            last_value = 0.0  # 初始化上一轮的值
            for t in range(start, end + 1):
                key = str(t)
                if key in data_dict:
                    last_value = data_dict[key]  # 更新上一轮的值
                    values.append(last_value)
                else:
                    values.append(last_value)  # 使用上一轮的值
            return values
        
        passk_values = extract_series(passk_dict)
        pass1_values = extract_series(pass1_dict)
        
        min_passk = min(passk_values)
        adjusted_passk = [value - min_passk for value in passk_values]
        passk_auc.append(max(np.trapz(adjusted_passk, dx=1), 0.0))
        
        min_pass1 = min(pass1_values)
        adjusted_pass1 = [value - min_pass1 for value in pass1_values]
        pass1_auc.append(max(np.trapz(adjusted_pass1, dx=1), 0.0))
        
        def get_value_at_step(data_dict):
            end_key = str(end)
            if end_key in data_dict:
                return data_dict[end_key]
            if not data_dict:
                return np.nan
            numeric_keys = sorted(int(k) for k in data_dict.keys())
            lower_keys = [k for k in numeric_keys if k <= end]
            if lower_keys:
                return data_dict[str(lower_keys[-1])]
            higher_keys = [k for k in numeric_keys if k > end]
            if higher_keys:
                return data_dict[str(higher_keys[0])]
            return np.nan
        
        passk_end_values.append(get_value_at_step(passk_dict))
        pass1_end_values.append(get_value_at_step(pass1_dict))
    
    span = max(end - start, 1)  # 避免除零
    df['passk_auc'] = np.array(passk_auc, dtype=float) / span
    df['pass1_auc'] = np.array(pass1_auc, dtype=float) / span
    df['pass_k'] = passk_end_values
    df['pass_mean'] = pass1_end_values
    
    return df

In [ ]:
task = "webshop"
# format = "user_assistant"
format = "user_assistant_format_part"


In [ ]:
# 设置存放所有实验结果的根目录
all_exp_base_dir = f"res_fake/{task}/"

# 加载所有实验结果到 DataFrame
df_experiments = load_all_results(all_exp_base_dir)

# 显示 DataFrame 的前几行以供检查
if not df_experiments.empty:
    print(f"成功加载 {len(df_experiments)} 个实验的结果。")
    # display(df_experiments.head())
else:
    print("未能加载任何实验结果。")

In [ ]:
selection_criteria = {
    # 'model_name': 'Qwen3-4B',
    # "enable_thinking": False,
    "state": "env",
    'chat_format': f'{format}',
    # "alfworld_mode": "eval_in_distribution",
    'history_has_cot': True,
    "stop_by_self": False,
    # "offer_feedback": True,
    # "prompt_example": "fewshot",
    # "history_window_size": 0,
}
# if format == "user_assistant_format_part":
#     selection_criteria['history_window_size'] = 1

In [ ]:
# --- 2. 根据条件筛选 DataFrame ---
selected_df = df_experiments
if selection_criteria:
    valid_criteria = {}
    for key, value in selection_criteria.items():
        if key in df_experiments.columns:
            valid_criteria[key] = value
        else:
            print(f"警告: DataFrame 中未找到列 '{key}'，已跳过该筛选条件。")
    if valid_criteria:
        query_str = " & ".join(
        [f"`{k}` == {repr(v)}" for k, v in valid_criteria.items()]
        )
        selected_df = df_experiments.query(query_str)
    else:
        print("提示: 所有筛选条件均被忽略，使用完整数据集。")
sort_columns = []
sort_orders = []
# if 'pass_mean' in selected_df.columns:
#     sort_columns.append('pass_mean')
#     sort_orders.append(False)  # pass_mean 降序排，便于查看表现最好的一组
model_order = [
    'Qwen3-4B',
    'Qwen3-30B-A3B',
    'Llama3-8B',
    'Llama3-70B',
    'Glm-9B-Chat',
    'Glm4-9B-Chat',
    "GLM-4-32B-0414",
    'Mistral-7B-Instruct-v0.3',
    'Ministral-3-14B-Instruct-2512',
    'phi-4',
    'deepseek-v3',
    'deepseek-v3.2',
    'gemini-2.5-flash',
    'gemini-2.5-flash-nothinking',
    'Phi-4-reasoning',
    'gpt-oss-120b',
    'deepseek-r1',
    'gemini-2.5-pro',
]
# 创建一个映射字典,将模型名称映射到排序索引
model_order_map = {model: idx for idx, model in enumerate(model_order)}
fallback_sort = ['model_order_map','chat_format','enable_thinking','history_has_cot','state']
selected_df['model_sort_order'] = selected_df['model_name'].map(
    lambda x: model_order_map.get(x, 999)  # 未在列表中的模型排在最后
)
sort_list = ['enable_thinking','model_sort_order','chat_format','history_has_cot',"state"]
selected_df = selected_df.sort_values(by=sort_list, ascending=[True, True, True, True, False]).reset_index(drop=True)
print(f"根据筛选条件，共找到 {len(selected_df)} 个实验。")
display(
    selected_df[["model_name","state", "chat_format","enable_thinking", "history_has_cot", "pass_mean", "mean_loop_ratio_after_invalid_steps_stepnorm"]]
)


In [ ]:
start_end_mapping = {
    "blocksworld": (0, 20),
    "frozenlake": (0, 30),
    "sodoku": (0, 20),
    "alfworld": (0, 60),
    "webshop": (0, 15)
}
start, end = start_end_mapping.get(task)
selected_df = cal_area_under_curve(selected_df, start=start, end=end)
selected_df["pass_mean_at_end"] = selected_df["pass1_at_T"].apply(lambda data: data.get(str(end), np.nan))
# display(
#     selected_df[["experiment", "pass_mean_at_end", "pass1_auc", "mean_loop_ratio_after_invalid_steps_stepnorm"]]
#  )

In [ ]:
from utils.analysis_files.analysis import get_config_label
exclude_keys = set(selection_criteria.keys()) if selection_criteria else set()
print("MEAN PASS\tAUV")
for idx, row in selected_df.iterrows():
    label = get_config_label(row, exclude_keys=exclude_keys)
    # print(label)
    # pass_k = row["pass_k"]
    pass_mean = row["pass_mean"]
    loop_ratio_steplevel = row["mean_loop_ratio_after_invalid_steps_stepnorm"]
    # loop_ratio_trajlevel = row["mean_loop_ratio_after_invalid_steps_trajnorm"]
    auv = row["pass1_auc"]
    
    print(f"{pass_mean*100:.1f}\t{auv*100:.1f}")

In [ ]:
print("AUV\t LR")
for idx, row in selected_df.iterrows():
    label = get_config_label(row, exclude_keys=exclude_keys)
    # print(label)
    # pass_k = row["pass_k"]
    pass_mean = row["pass_mean"]
    loop_ratio_steplevel = row["mean_loop_ratio_after_invalid_steps_stepnorm"]
    # loop_ratio_trajlevel = row["mean_loop_ratio_after_invalid_steps_trajnorm"]
    auv = row["pass1_auc"]
    
    print(f"{auv*100:.1f}\t{loop_ratio_steplevel*100:.1f}")